# Quickr ? PySpark RDD MapReduce Analysis

This is the **primary Phase 3 implementation**. Run cells in order with **Run All** in VS Code.
Select the Python interpreter at `../.venv-spark/Scripts/python.exe` (Python 3.11) using **Select Kernel ? Python Environments**.
The notebook works with a kernel working directory at the project root or `notebooks/`.

The input is `data/processed/quickr_clean.csv`. Only small aggregate results are collected to the driver and exported to `data/analytics/` for later Cassandra integration. No database is used here.

Spark executes locally with two worker threads; this demonstrates partitioned MapReduce, not a distributed cluster. The saved code-cell outputs below come from executing this notebook.

Dependencies: `pyspark`, `ipykernel`, `nbclient`, and `nbformat` in `.venv-spark`. Python 3.12 workers crashed on Windows in this project; use Python 3.11. Java 8u202 works here, but is deprecated for Spark 3.5.

## 1. Setup, portable paths and source protection

Resolve the project from the current directory or its parents. Record SHA-256 hashes to verify that both datasets stay unchanged. Use the kernel's Python for Spark workers. Windows launcher settings are corrected only inside this process.

In [1]:
import csv
import hashlib
import math
import os
from pathlib import Path
import sys
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/quickr_clean.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook within the BDA LA workspace.')
SOURCE = ROOT / 'data/processed/quickr_clean.csv'
DESTINATION = ROOT / 'data/analytics'
REPORT = ROOT / 'reports/spark_analytics.md'
PARTITIONS = 2
protected = [SOURCE, ROOT / 'data/raw/food_delivery.csv']
hashes = {p: hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}

if os.name == 'nt' and sys.version_info >= (3, 12):
    raise RuntimeError('Select the Python 3.11 .venv-spark kernel on Windows.')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
import pyspark
from pyspark.sql import SparkSession
os.environ['SPARK_HOME'] = str(Path(pyspark.__file__).resolve().parent)
hadoop_home = Path(os.environ.get('HADOOP_HOME', '__not_configured__'))
if hadoop_home.name.lower() == 'bin' and (hadoop_home / 'winutils.exe').is_file():
    os.environ['HADOOP_HOME'] = str(hadoop_home.parent)
print('Workspace:', ROOT)
print('Kernel:', sys.executable)
print('Python:', sys.version.split()[0], '| PySpark:', pyspark.__version__)

Workspace: C:\Users\AAROHI\Desktop\5th sem\BDA LAB\BDA LA
Kernel: C:\Users\AAROHI\Desktop\5th sem\BDA LAB\BDA LA\.venv-spark\Scripts\python.exe
Python: 3.11.17 | PySpark: 3.5.7


## 2. Start Spark and inspect the actual schema

The Spark CSV reader handles headers and quoting. CSV fields initially remain strings; later RDD `map` operations explicitly parse the numerical values needed for analytics. No DataFrame or pandas groupby is used.

In [2]:
spark = (SparkSession.builder.master("local[2]").appName("Quickr-RDD-Analytics")
         .config("spark.jars", (ROOT / ".spark-packages/spark-cassandra-connector-assembly_2.12-3.5.1.jar").as_uri())
         .config("spark.ui.enabled", "false")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.default.parallelism", str(PARTITIONS))
         .config("spark.python.worker.reuse", "true")
         .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
from pyspark.sql import functions as F
with SOURCE.open(newline='', encoding='utf-8') as handle:
    source_columns = next(csv.reader(handle))
frame = (spark.read.format('org.apache.spark.sql.cassandra')
         .options(keyspace='quickr_bda', table='cleaned_delivery_records',
                  **{'spark.cassandra.connection.host': '127.0.0.1',
                     'spark.cassandra.connection.port': '9042'}).load()
         .select(*[F.col(c).cast('string').alias(c) for c in source_columns]))
required = {'duration_min', 'traffic', 'weather', 'city', 'vehicle_type',
            'festival', 'order_type', 'distance_km', 'delivery_person_age'}
if required - set(frame.columns):
    raise ValueError(f'Missing columns: {sorted(required - set(frame.columns))}')
frame.printSchema()
frame.show(5, truncate=False)
java_version = spark.sparkContext._jvm.java.lang.System.getProperty('java.version')
print('Spark:', spark.version, '| Java:', java_version, '| Master:', spark.sparkContext.master)

root
 |-- order_id: string (nullable = true)
 |-- delivery_person_id: string (nullable = true)
 |-- delivery_person_age: string (nullable = true)
 |-- delivery_person_rating: string (nullable = true)
 |-- restaurant_latitude: string (nullable = true)
 |-- restaurant_longitude: string (nullable = true)
 |-- delivery_latitude: string (nullable = true)
 |-- delivery_longitude: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_time: string (nullable = true)
 |-- pickup_time: string (nullable = true)
 |-- weather: string (nullable = true)
 |-- traffic: string (nullable = true)
 |-- vehicle_condition: string (nullable = true)
 |-- order_type: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- multiple_deliveries: string (nullable = true)
 |-- festival: string (nullable = true)
 |-- city: string (nullable = true)
 |-- duration_min: string (nullable = true)
 |-- delivery_person_age_invalid: string (nullable = true)
 |-- delivery_person_rating_

+--------+------------------+-------------------+----------------------+-------------------+--------------------+-----------------+------------------+----------+----------+-----------+----------+-------+-----------------+----------+------------+-------------------+--------+------------+------------+---------------------------+------------------------------+---------------------------+-------------------------+---------------+----------+-----------+-----------------+-----------+------------------+---------------------------+------------------------------+---------------------------+-------------------------+------------------+------------------+-------------------+
|order_id|delivery_person_id|delivery_person_age|delivery_person_rating|restaurant_latitude|restaurant_longitude|delivery_latitude|delivery_longitude|order_date|order_time|pickup_time|weather   |traffic|vehicle_condition|order_type|vehicle_type|multiple_deliveries|festival|city        |duration_min|delivery_person_age_invalid

## 3. Safe parsing, bands and the MapReduce helper

`map` transforms each delivery into `(category, (duration, 1))`. `reduceByKey` shuffles matching categories together and combines their duration sums and counts. The final `map` computes `sum / count`, and `collect` returns only the small summary table.

For an illustrative pair of durations 20 and 30, the reducer produces `(50, 2)` and the average is 25 minutes. This is an example, not a dataset finding.

Missing categories and missing ages use `unknown`, so deliveries are not silently dropped. Distance intervals include the lower bound and exclude the upper bound. Ages under 18 are retained in a review band.

In [3]:
def number(value):
    """Return a finite float, or None for missing/malformed input."""
    try:
        result = float(value)
        return result if math.isfinite(result) else None
    except (ValueError, TypeError):
        return None


def category(value):
    if value is None or str(value).strip().lower() in {"", "nan", "null", "none"}:
        return "unknown"
    return str(value).strip().lower()


def distance_band(value):
    distance = number(value)
    if distance is None or distance < 0:
        return "unknown"
    if distance < 5:
        return "[0,5) km"
    if distance < 10:
        return "[5,10) km"
    if distance < 15:
        return "[10,15) km"
    if distance < 20:
        return "[15,20) km"
    return "[20,+) km"


def age_band(value):
    age = number(value)
    if age is None or age <= 0 or age > 120:
        return "unknown"
    if age < 18:
        return "under_18_review"
    if age < 25:
        return "18-24"
    if age < 35:
        return "25-34"
    if age < 45:
        return "35-44"
    return "45_plus"


def combine(left, right):
    """Reducer combines partial duration sums and record counts."""
    return left[0] + right[0], left[1] + right[1]


def grouped_average(records, key_function):
    # map -> shuffle/reduceByKey -> map -> collect is the core MapReduce model.
    return sorted(records.map(lambda row: (key_function(row), (row["duration_min"], 1)))
                  .reduceByKey(combine, numPartitions=PARTITIONS)
                  .map(lambda item: (item[0], item[1][1], item[1][0],
                                     item[1][0] / item[1][1]))
                  .collect())


def export_csv(filename, header, rows):
    path = DESTINATION / filename
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(header)
        writer.writerows(rows)
    # Verify the serialized file too, rather than just trusting in-memory results.
    with path.open(newline="", encoding="utf-8") as handle:
        saved = list(csv.DictReader(handle))
    if len(saved) != len(rows):
        raise ValueError(f"CSV row count mismatch: {path}")
    return saved



def show_table(headers, rows):
    # Markdown tables display in VS Code without pandas or extra dependencies.
    text = '| ' + ' | '.join(headers) + ' |\n'
    text += '| ' + ' | '.join(['---'] * len(headers)) + ' |\n'
    for row in rows:
        values = [f'{v:.6f}' if isinstance(v, float) else str(v) for v in row]
        text += '| ' + ' | '.join(values) + ' |\n'
    display(Markdown(text))

grouped = {}
print('Helpers ready. Distance bands: [0,5), [5,10), [10,15), [15,20), [20,+) km.')

Helpers ready. Distance bands: [0,5), [5,10), [10,15), [15,20), [20,+) km.


## 4. Build and cache RDDs; filter unusable durations

`repartition(2)` distributes records into two partitions. `map` converts Spark Rows into Python dictionaries and parses duration. `filter` retains positive, finite durations. `cache` supports repeated analytics, and the `count` actions materialize both RDDs. All average denominators use valid-duration records; order counts use every delivery.

In [4]:
all_records = frame.rdd.repartition(PARTITIONS).map(lambda row: row.asDict()).cache()
total = all_records.count()
valid = (all_records.map(lambda row: dict(row, duration_min=number(row["duration_min"])))
         .filter(lambda row: row["duration_min"] is not None and row["duration_min"] > 0)
         .cache())
valid_count = valid.count()
if not total or not valid_count:
    raise ValueError("No deliveries with a valid duration; cannot calculate averages.")

print('Total deliveries:', total)
print('Valid-duration deliveries:', valid_count)
print('Invalid-duration exclusions:', total - valid_count)
print('Record partitions:', all_records.getNumPartitions())
print('Valid-duration partitions:', valid.getNumPartitions())

Total deliveries: 41522
Valid-duration deliveries: 41522
Invalid-duration exclusions: 0
Record partitions: 2
Valid-duration partitions: 2


## 5. Total deliveries and mean delivery duration

`map` emits `(duration, 1)`; `reduce` adds all pairs. Divide the resulting sum by its count. This is a global MapReduce aggregation with a checked, positive denominator.

In [5]:
sums = valid.map(lambda row: (row['duration_min'], 1)).reduce(combine)
overall_average = sums[0] / sums[1]
show_table(['Total deliveries', 'Valid durations', 'Total duration (min)', 'Average (min)'],
           [(total, valid_count, sums[0], overall_average)])

| Total deliveries | Valid durations | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| 41522 | 41522 | 1092670.000000 | 26.315447 |


## 6. Average duration by traffic density

Compare the observed traffic categories, including unknown. The helper performs `map ? reduceByKey ? map ? collect`. Each output includes count and sum so its denominator is visible.

In [6]:
grouped['traffic'] = grouped_average(valid, lambda row: category(row['traffic']))
show_table(['Traffic Density', 'Deliveries', 'Total duration (min)', 'Average (min)'], grouped['traffic'])

| Traffic Density | Deliveries | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| high | 4046 | 110098.000000 | 27.211567 |
| jam | 12950 | 403892.000000 | 31.188571 |
| low | 14095 | 299953.000000 | 21.280809 |
| medium | 10023 | 267979.000000 | 26.736406 |
| unknown | 408 | 10748.000000 | 26.343137 |


## 7. Average duration by weather condition

Compare normalized weather categories, retaining unknown weather. The helper performs `map ? reduceByKey ? map ? collect`. Each output includes count and sum so its denominator is visible.

In [7]:
grouped['weather'] = grouped_average(valid, lambda row: category(row['weather']))
show_table(['Weather Condition', 'Deliveries', 'Total duration (min)', 'Average (min)'], grouped['weather'])

| Weather Condition | Deliveries | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| cloudy | 6881 | 199078.000000 | 28.931551 |
| fog | 6962 | 201848.000000 | 28.992818 |
| sandstorms | 6853 | 177340.000000 | 25.877718 |
| stormy | 6932 | 179484.000000 | 25.892095 |
| sunny | 6682 | 146303.000000 | 21.895091 |
| unknown | 420 | 11082.000000 | 26.385714 |
| windy | 6792 | 177535.000000 | 26.138840 |


## 8. Average duration by city category

City denotes settlement type, not a city name. Small groups need cautious interpretation. The helper performs `map ? reduceByKey ? map ? collect`. Each output includes count and sum so its denominator is visible.

In [8]:
grouped['city'] = grouped_average(valid, lambda row: category(row['city']))
show_table(['City Category', 'Deliveries', 'Total duration (min)', 'Average (min)'], grouped['city'])

| City Category | Deliveries | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| metropolitan | 31103 | 849853.000000 | 27.323827 |
| semi_urban | 147 | 7303.000000 | 49.680272 |
| unknown | 1097 | 24252.000000 | 22.107566 |
| urban | 9175 | 211262.000000 | 23.025831 |


## 9. Average duration by vehicle type

Compare vehicle categories; these descriptive means do not establish causal effects. The helper performs `map ? reduceByKey ? map ? collect`. Each output includes count and sum so its denominator is visible.

In [9]:
grouped['vehicle_type'] = grouped_average(valid, lambda row: category(row['vehicle_type']))
show_table(['Vehicle Type', 'Deliveries', 'Total duration (min)', 'Average (min)'], grouped['vehicle_type'])

| Vehicle Type | Deliveries | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| bicycle | 43 | 1116.000000 | 25.953488 |
| electric_scooter | 3404 | 83293.000000 | 24.469154 |
| motorcycle | 24197 | 668105.000000 | 27.611067 |
| scooter | 13878 | 340156.000000 | 24.510448 |


## 10. Average duration by festival status

Keep yes, no and unknown separate. Unknown is not treated as no. The helper performs `map ? reduceByKey ? map ? collect`. Each output includes count and sum so its denominator is visible.

In [10]:
grouped['festival'] = grouped_average(valid, lambda row: category(row['festival']))
show_table(['Festival Status', 'Deliveries', 'Total duration (min)', 'Average (min)'], grouped['festival'])

| Festival Status | Deliveries | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| no | 40495 | 1053353.000000 | 26.011927 |
| unknown | 215 | 2401.000000 | 11.167442 |
| yes | 812 | 36916.000000 | 45.463054 |


## 11. Delivery counts by order type

`map` emits `(order_type, 1)`. `reduceByKey` adds counts and `collect` retrieves the small count summary. Unlike duration averages, this count includes all records regardless of duration validity.

In [11]:
order_counts = sorted(all_records.map(lambda row: (category(row['order_type']), 1))
                      .reduceByKey(lambda a, b: a + b, numPartitions=PARTITIONS).collect())
show_table(['Order type', 'Deliveries'], order_counts)

| Order type | Deliveries |
| --- | --- |
| buffet | 10265 |
| drinks | 10348 |
| meal | 10398 |
| snack | 10511 |


## 12. Average duration by distance band

`map` assigns the Haversine distance to an explicit interval before creating `(band, (duration, 1))`. The same reducer combines sums and counts. Straight-line distance is not road distance.

In [12]:
grouped['distance_band'] = grouped_average(valid, lambda row: distance_band(row['distance_km']))
show_table(['Distance band', 'Deliveries', 'Total duration (min)', 'Average (min)'], grouped['distance_band'])

| Distance band | Deliveries | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| [0,5) km | 11187 | 248182.000000 | 22.184857 |
| [10,15) km | 11183 | 333716.000000 | 29.841366 |
| [15,20) km | 6186 | 184945.000000 | 29.897349 |
| [20,+) km | 1779 | 52987.000000 | 29.784711 |
| [5,10) km | 11187 | 272840.000000 | 24.389023 |


## 13. Average duration by delivery-person age band

`map` assigns an age band, including `unknown` and `under_18_review`. `reduceByKey` combines sums/counts, followed by average calculation and collection. No age is imputed from duration.

In [13]:
grouped['age_band'] = grouped_average(valid, lambda row: age_band(row['delivery_person_age']))
show_table(['Age band', 'Deliveries', 'Total duration (min)', 'Average (min)'], grouped['age_band'])

| Age band | Deliveries | Total duration (min) | Average (min) |
| --- | --- | --- | --- |
| 18-24 | 9905 | 228044.000000 | 23.023120 |
| 25-34 | 19901 | 523594.000000 | 26.309934 |
| 35-44 | 10191 | 300890.000000 | 29.525071 |
| 45_plus | 24 | 539.000000 | 22.458333 |
| under_18_review | 25 | 640.000000 | 25.600000 |
| unknown | 1476 | 38963.000000 | 26.397696 |


## 14. Deliveries exceeding a 30-minute analytical threshold

`filter` selects durations **strictly greater than 30**, and `count` produces the numerator. Divide by all valid-duration deliveries and multiply by 100. This is **not actual lateness**: the data supplies no promised deadline or service-level agreement.

In [14]:
exceeding = valid.filter(lambda row: row['duration_min'] > 30).count()
percentage = exceeding * 100 / valid_count
assert 0 <= percentage <= 100
show_table(['Threshold (min)', 'Exceeding', 'Denominator', 'Percentage'],
           [(30, exceeding, valid_count, percentage)])

| Threshold (min) | Exceeding | Denominator | Percentage |
| --- | --- | --- | --- |
| 30 | 12443 | 41522 | 29.967246 |


## 15. Export and verify all nine CSV summaries

Export collected aggregates using Python's CSV writer. Each group must reconcile with the overall count and duration sum. Read exported CSVs back and check counts and average formulas. These files retain stable names and headers for later Cassandra integration; no Cassandra connection is created.

In [15]:
for name, rows in grouped.items():
    assert sum(row[1] for row in rows) == valid_count, name
    assert math.isclose(sum(row[2] for row in rows), sums[0]), name
    assert all(row[1] > 0 and math.isfinite(row[3]) for row in rows), name
assert sum(count for _, count in order_counts) == total
assert 0 <= percentage <= 100
DESTINATION.mkdir(parents=True, exist_ok=True)
summary = [(total, valid_count, total - valid_count, sums[0], overall_average,
            30, exceeding, percentage)]
export_csv("overall_summary.csv", ["total_deliveries", "valid_duration_deliveries",
           "excluded_invalid_duration", "total_duration_min", "average_duration_min",
           "analytical_threshold_min", "deliveries_exceeding_threshold",
           "percent_exceeding_threshold"], summary)
for name, rows in grouped.items():
    saved = export_csv(f"{name}_summary.csv", [name, "delivery_count", "total_duration_min",
                       "average_duration_min"], rows)
    assert sum(int(row["delivery_count"]) for row in saved) == valid_count
    assert all(math.isclose(float(row["average_duration_min"]),
               float(row["total_duration_min"]) / int(row["delivery_count"])) for row in saved)
export_csv("order_type_summary.csv", ["order_type", "delivery_count"], order_counts)
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == h for p, h in hashes.items())

for path in sorted(DESTINATION.glob('*_summary.csv')):
    with path.open(newline='', encoding='utf-8') as handle:
        exported_rows = list(csv.DictReader(handle))
    if 'delivery_count' in exported_rows[0]:
        assert sum(int(row['delivery_count']) for row in exported_rows) == total
    print(path.name, ':', len(exported_rows), 'aggregate rows verified')
print('Raw and cleaned CSV hashes unchanged.')

age_band_summary.csv : 6 aggregate rows verified
city_summary.csv : 4 aggregate rows verified
distance_band_summary.csv : 5 aggregate rows verified
festival_summary.csv : 3 aggregate rows verified
order_type_summary.csv : 4 aggregate rows verified
overall_summary.csv : 1 aggregate rows verified
traffic_summary.csv : 5 aggregate rows verified
vehicle_type_summary.csv : 4 aggregate rows verified
weather_summary.csv : 7 aggregate rows verified
Raw and cleaned CSV hashes unchanged.


## 16. Regenerate the Phase 3 report

Write the actual findings, operation explanations, input schema, execution settings and limitations to `reports/spark_analytics.md`. Historical troubleshooting remains documented; the notebook is now the primary implementation.

In [16]:
java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")
report = ["# Quickr - PySpark RDD MapReduce Analytics\n",
          "## Purpose and execution\n",
          "PySpark partitions records and computes aggregates through RDD transformations "
          "and actions. CSV parsing uses Spark's reader; the core analytics use real "
          "Python RDD operations, with no pandas or SQL groupby.\n",
          f"Spark/PySpark: {spark.version}; Java: {java_version}; Python: {sys.version.split()[0]}. "
          f"Master: `{spark.sparkContext.master}`; cached record RDD partitions: "
          f"{all_records.getNumPartitions()}; valid-duration RDD partitions: "
          f"{valid.getNumPartitions()}; reduceByKey partitions: {PARTITIONS}.\n",
          "This is local execution with two worker threads on one Windows computer, "
          "not a distributed cluster. Spark UI is disabled; workers use the active "
          "virtual environment and the driver binds to loopback. "
          "Open `notebooks/Quickr_PySpark_Analysis.ipynb`, select the Python 3.11 .venv-spark kernel and Run All. "
          "The separate Python 3.11 environment contains PySpark/Py4J and Jupyter execution dependencies; "
          "the original Phase 1/2 environment is retained.\n",
          "## Input schema\n",
          "CSV fields are read as nullable strings, then required numeric values are "
          "parsed explicitly in RDD transformations. Original column names:\n",
          "```text\n" + "\n".join(frame.columns) + "\n```\n",
          "## MapReduce model\n",
          "Each grouped average maps a record to `(category, (duration, 1))`. "
          "`reduceByKey` combines partial `(sum, count)` pairs across partitions; "
          "a final `map` divides the duration sum by its positive count. "
          "For illustration only, two same-category durations 20 and 30 map to "
          "`(category, (20, 1))` and `(category, (30, 1))`, reduce to `(50, 2)`, "
          "then yield 25 minutes. These example numbers are not dataset findings.\n",
          "Transformations: `map`, `filter`, `repartition`, `reduceByKey`; "
          "`cache` avoids repeatedly loading records. Actions: `count`, `reduce`, "
          "and `collect`. Only small aggregate results are collected to the driver.\n",
          "## Actual overall findings\n",
          f"Total deliveries: **{total:,}**. Valid positive finite durations: **{valid_count:,}**. "
          f"Excluded from duration calculations: **{total - valid_count:,}**. "
          f"Average duration: **{overall_average:.6f} minutes**.\n",
          f"**{exceeding:,} / {valid_count:,} ({percentage:.6f}%)** exceed 30 minutes. "
          "This is an analytical threshold, not actual lateness: no promised delivery "
          "deadline or service-level agreement is available.\n"]
for name, rows in grouped.items():
    report += [f"## Average duration by {name.replace('_', ' ')}\n",
               f"Output: `data/analytics/{name}_summary.csv`.\n",
               "| Category | Deliveries | Total duration (min) | Average (min) |\n"
               "|---|---:|---:|---:|\n" + "\n".join(
                   f"| {key} | {count} | {duration:.6f} | {average:.6f} |"
                   for key, count, duration, average in rows) + "\n"]
report += ["## Delivery counts by order type\n",
           "| Order type | Deliveries |\n|---|---:|\n" + "\n".join(
               f"| {key} | {count} |" for key, count in order_counts) + "\n",
           "## Null handling, bands and limitations\n",
           "Missing categories are grouped as unknown. Missing/invalid age or distance "
           "uses an unknown band, retaining a valid-duration delivery in each denominator. "
           "Distance bands are [0,5), [5,10), [10,15), [15,20), [20,+) km; "
           "age bands are under 18 (review), 18-24, 25-34, 35-44, 45+, and unknown. "
           "Grouped means use only valid positive finite durations; order-type counts "
           "use all records. Groups with no records are absent, never divided by zero.\n",
           "These are descriptive associations, not causal effects. Small categories "
           "can have unstable averages. Cleaning exclusions and unknown inputs affect "
           "representativeness. Haversine distance is straight-line, not road distance. "
           "CSV exports use the Python standard library after collecting small summaries; "
           "this avoids requiring Hadoop native writers on Windows.\n",
           "## Environment and verification\n",
           "Initial installation stalled without output and was stopped; a retry "
           "downloaded and installed PySpark successfully. The first Spark run failed "
           "with `[PATH_NOT_FOUND]` because the file URI encoded workspace spaces as "
           "`%20`; using a forward-slash Windows path fixed it. "
           "The existing HADOOP_HOME pointed to C:/hadoop/bin, producing a missing "
           "C:/hadoop/bin/bin warning. The notebook verifies winutils.exe and corrects "
           "HADOOP_HOME to its parent for this process only. SPARK_HOME is set explicitly "
           "to avoid Windows launcher Python discovery with spaces.\n",
           "Python 3.12 runs then failed with `Python worker exited unexpectedly "
           "(crashed)` caused by `java.io.EOFException`, even outside the sandbox. "
           "This matches [SPARK-53759](https://issues.apache.org/jira/browse/SPARK-53759). "
           "A fresh Python 3.11 virtual environment was created using an available "
           "interpreter, without importing any previous project code. An offline "
           "package-cache attempt hit `[Errno 13] Permission denied`; installation "
           "was retried with approved access. Python 3.11 pip then reported "
           "`SSL: CERTIFICATE_VERIFY_FAILED`. A binary-only public PySpark download "
           "also reported no matching distribution (PySpark is distributed as source). "
           "The successfully built local PySpark wheel and a Py4J wheel downloaded "
           "through the working Python 3.12 pip were staged in `.spark-packages` "
           "and installed offline into `.venv-spark`, without disabling TLS checks. "
           "Historical failures are preserved in "
           "spark_execution.log, spark_execution_retry.log and "
           "spark_execution_unrestricted.log. The final run is spark_execution_final.log.\n",
           "Installed PySpark 3.5.7 and its Py4J dependency. Java 8u202 was already "
           "installed. Java 8 versions before 8u371 are deprecated for Spark 3.5; "
           "a current Java 17 installation is preferable for future work. "
           "See [Apache Spark requirements](https://spark.apache.org/docs/3.5.7/).\n",
           "Remaining environment messages in the successful run: the Windows launcher "
           "printed `The system cannot find the path specified` before Spark started; "
           "the source of that shell message is unresolved. In the earlier script run optional psutil was absent, "
           "so Spark warns about reduced memory-spill monitoring. After verified outputs "
           "and Spark shutdown, `ERROR: Access denied` was printed; its exact cleanup "
           "operation is unresolved. The process nevertheless exited with code 0. "
           "These messages are retained in the final execution log; no result failures "
           "or incomplete aggregate outputs were observed.\n",
           "All actions completed. Exported CSV counts and grouped averages were read "
           "back and checked; each grouping reconciles with the overall duration count "
           "and sum. Raw and cleaned CSV hashes are unchanged. No dedicated test suite "
           "or later project phases were implemented.\n"]
REPORT.write_text("\n".join(report), encoding="utf-8")

with REPORT.open('a', encoding='utf-8') as handle:
    handle.write('\n## Notebook implementation\n\n'
                 'The primary implementation is notebooks/Quickr_PySpark_Analysis.ipynb. '
                 'All analysis code is contained in its labeled cells; no external analytics '
                 'script is imported. Saved cell outputs come from notebook execution. '
                 'Use the Python 3.11 .venv-spark interpreter in VS Code and Run All. '
                 'The earlier standalone spark_analytics.py has been replaced. '
                 'Earlier log messages in the environment section are historical; '
                 'the executed notebook outputs record this run.\n')
print('Updated:', REPORT.relative_to(ROOT))

Updated: reports\spark_analytics.md


## 17. Findings, limitations and shutdown

Review the executed tables above for actual comparisons. Small groups (such as bicycle and semi-urban) can have unstable averages. These are descriptive associations. Cleaning exclusions affect representativeness; missing age remains an unknown band. Local Spark execution does not demonstrate cluster scale.

The cleanup cell releases cached RDDs and stops Spark. After running it, use **Restart Kernel ? Run All** for a fresh complete execution. No ML, Cassandra or application is implemented.

In [17]:
print(f'Analyzed {total:,} deliveries; mean duration {overall_average:.6f} minutes.')
print(f'Over 30 minutes: {exceeding:,}/{valid_count:,} = {percentage:.6f}% (analytical threshold).')
valid.unpersist()
all_records.unpersist()
spark.stop()
print('Spark stopped. All nine summaries are exported and verified.')

Analyzed 41,522 deliveries; mean duration 26.315447 minutes.
Over 30 minutes: 12,443/41,522 = 29.967246% (analytical threshold).


Spark stopped. All nine summaries are exported and verified.
